# Vivir más sin ser más rico — armado de los datos

Este notebook descarga de Data360 los indicadores que usamos en la nota, los limpia y los deja en un panel país–año en `../datos/panel.csv`.

| Variable | Indicador (WDI) | Unidad |
|---|---|---|
| Esperanza de vida al nacer | `WB_WDI_SP_DYN_LE00_IN` | años |
| PIB per cápita, PPA | `WB_WDI_NY_GDP_PCAP_PP_KD` | dólares internacionales constantes de 2021 |
| Población | `WB_WDI_SP_POP_TOTL` | personas |
| Gasto corriente en salud per cápita, PPA | `WB_WDI_SH_XPD_CHEX_PP_CD` | dólares internacionales corrientes |
| Mortalidad de menores de 5 años | `WB_WDI_SH_DYN_MORT` | por cada 1.000 nacidos vivos |

Usamos el PIB en **paridad de poder adquisitivo (PPA)** porque compara mejor el nivel de vida entre países: un dólar no compra lo mismo en Costa Rica que en Estados Unidos.

In [1]:
# !pip install babel pycountry   # descomentar en Colab
import requests
import numpy as np
import pandas as pd
import pycountry
from babel import Locale

BASE = "https://data360api.worldbank.org/data360"
INDICADORES = {
    "vida": "WB_WDI_SP_DYN_LE00_IN",
    "pib": "WB_WDI_NY_GDP_PCAP_PP_KD",
    "poblacion": "WB_WDI_SP_POP_TOTL",
    "gasto_salud": "WB_WDI_SH_XPD_CHEX_PP_CD",
    "mortalidad_5": "WB_WDI_SH_DYN_MORT",
}
DESDE = 1990


In [2]:
def traer_datos(base, indicador, **filtros):
    filas, skip = [], 0
    while True:
        params = {"DATABASE_ID": base, "INDICATOR": indicador, "top": 1000, "skip": skip, **filtros}
        r = requests.get(f"{BASE}/data", params=params, timeout=120)
        r.raise_for_status()
        pagina = r.json()
        filas += pagina["value"]
        skip += 1000
        if skip >= pagina["count"]:
            return pd.DataFrame(filas)


def serie(nombre, codigo):
    d = traer_datos("WB_WDI", codigo)
    d = d[d["SEX"].isin(["_T", None]) & d["AGE"].isin(["_T", None])]
    return pd.DataFrame({
        "iso3": d["REF_AREA"],
        "anio": d["TIME_PERIOD"].astype(int),
        nombre: pd.to_numeric(d["OBS_VALUE"], errors="coerce"),
    })


series = {nombre: serie(nombre, codigo) for nombre, codigo in INDICADORES.items()}
for nombre, d in series.items():
    print(f"{nombre:13} {len(d):6} filas  {d['anio'].min()}–{d['anio'].max()}")


vida           17191 filas  1960–2024
pib             8705 filas  1990–2025
poblacion      17460 filas  1960–2025
gasto_salud     5715 filas  2000–2024
mortalidad_5   13541 filas  1960–2024


## Países, regiones y nivel de ingreso

La API mezcla países con agregados (`WLD`, `LCN`, `HIC`…). Nos quedamos solo con las economías que aparecen en la jerarquía de regiones del Banco Mundial y les agregamos el nombre en castellano.

In [3]:
jer = requests.get(f"{BASE}/portal/v1/hierarchy", params={"type": "country"}, timeout=60).json()["data"]

REGIONES = {
    "East Asia & Pacific": "Asia oriental y Pacífico",
    "Europe & Central Asia": "Europa y Asia central",
    "Latin America & Caribbean": "América Latina y el Caribe",
    "Middle East, North Africa, Afghanistan & Pakistan": "Medio Oriente, norte de África, Afganistán y Pakistán",
    "North America": "América del Norte",
    "South Asia": "Asia meridional",
    "Eastern and Southern Africa": "África subsahariana",
    "Western and Central Africa": "África subsahariana",
}
INGRESOS = {
    "Low income": "Bajo", "Lower middle income": "Medio bajo",
    "Upper middle income": "Medio alto", "High income": "Alto",
}

regiones = pd.DataFrame([{"iso3": c["id"], "nombre_en": c["name"], "region": REGIONES[g["name"]]}
                         for g in jer["HCL_WB_REGIONS"] for c in g["countries"]])
ingresos = pd.DataFrame([{"iso3": c["id"], "ingreso": INGRESOS.get(g["name"])}
                         for g in jer["HCL_WB_INCOME"] for c in g["countries"]])
paises = regiones.merge(ingresos, on="iso3", how="left")

nombres_es = Locale("es").territories
def nombre_es(fila):
    pais = pycountry.countries.get(alpha_3=fila.iso3)
    iso2 = pais.alpha_2 if pais else {"XKX": "XK"}.get(fila.iso3)
    return nombres_es.get(iso2, {"CHI": "Islas del Canal"}.get(fila.iso3, fila.nombre_en))

paises["pais"] = paises.apply(nombre_es, axis=1)
print(len(paises), "economías")
paises.sample(8, random_state=3)


220 economías


,iso3,nombre_en,region,ingreso,pais
130,RUS,Russian Federation,Europa y Asia central,Alto,Rusia
64,MYS,Malaysia,Asia oriental y Pacífico,Medio alto,Malasia
154,CHL,Chile,América Latina y el Caribe,Alto,Chile
86,ALB,Albania,Europa y Asia central,Medio alto,Albania
70,NCL,New Caledonia,Asia oriental y Pacífico,Alto,Nueva Caledonia
135,ESP,Spain,Europa y Asia central,Alto,España
34,GNQ,Equatorial Guinea,África subsahariana,Medio alto,Guinea Ecuatorial
95,CHI,Channel Islands,Europa y Asia central,Alto,Islas del Canal


Las dos regiones africanas del Banco Mundial las unificamos en «África subsahariana» para que el gráfico tenga menos colores y sea más fácil de leer.

In [4]:
panel = series["vida"]
for nombre in ["pib", "poblacion", "gasto_salud", "mortalidad_5"]:
    panel = panel.merge(series[nombre], on=["iso3", "anio"], how="outer")

panel = (panel[panel["anio"] >= DESDE]
         .merge(paises[["iso3", "pais", "region", "ingreso"]], on="iso3", how="inner")
         .sort_values(["iso3", "anio"], ignore_index=True))
print(panel.shape)
panel.sample(5, random_state=1)


(7812, 10)


,iso3,anio,vida,pib,poblacion,gasto_salud,mortalidad_5,pais,region,ingreso
420,AUT,2014,81.490244,60845.574398,8546356.0,5105.975608,3.8,Austria,Europa y Asia central,Alto
2341,FRO,1991,76.355573,NaN,47447.0,NaN,NaN,Islas Feroe,Europa y Asia central,Alto
5455,PAN,2009,76.228000,21822.996136,3564381.0,1043.113635,19.2,Panamá,América Latina y el Caribe,Alto
802,BLR,2000,68.912195,11182.454834,9979610.0,316.620795,12.8,Bielorrusia,Europa y Asia central,Medio alto
5403,PAK,1993,60.083000,3369.237959,127020117.0,NaN,131.5,Pakistán,"Medio Oriente, norte de África, Afganistán y P...",Medio bajo


## Cobertura

¿Cuántos países tienen a la vez esperanza de vida e ingreso en cada año? Necesitamos un año reciente con buena cobertura para el corte transversal.

In [5]:
cobertura = (panel.assign(ambos=panel["vida"].notna() & panel["pib"].notna(),
                          con_gasto=panel["gasto_salud"].notna())
             .groupby("anio")[["ambos", "con_gasto"]].sum())
cobertura.tail(8)


,ambos,con_gasto
anio,,
2018,199,192
2019,199,193
2020,199,193
2021,199,193
2022,199,192
2023,197,192
2024,195,22
2025,0,0


2024 es el último año con esperanza de vida e ingreso para casi todos los países. El gasto en salud llega hasta 2023, así que para esa variable tomamos el último dato disponible.

Algunos países quedan afuera del análisis porque **no publican PIB en PPA** reciente:

In [6]:
ultimo = panel[panel["anio"] == 2024]
sin_pib = ultimo[ultimo["vida"].notna() & ultimo["pib"].isna()]
print(len(sin_pib), "países con esperanza de vida pero sin PIB en 2024:")
print(", ".join(sorted(sin_pib["pais"])))


22 países con esperanza de vida pero sin PIB en 2024:
Corea del Norte, Cuba, Eritrea, Gibraltar, Groenlandia, Guam, Isla de Man, Islas Marianas del Norte, Islas Vírgenes Británicas, Islas Vírgenes de EE. UU., Islas del Canal, Liechtenstein, Mónaco, Nueva Caledonia, Polinesia Francesa, Samoa Americana, San Marino, San Martín, Siria, Sudán del Sur, Venezuela, Yemen


In [7]:
panel = panel.round({"vida": 2, "pib": 0, "gasto_salud": 0, "mortalidad_5": 1})
panel["poblacion"] = panel["poblacion"].astype("Int64")
panel.to_csv("../datos/panel.csv", index=False)
print("Guardado:", panel.shape)


Guardado: (7812, 10)
